In [1]:
import pandas as pd
from datetime import datetime, timedelta
import re

In [2]:
def clean_price(p):
    if not p:
        return None
    try:
        p = str(p).replace('Rs', '').replace(',', '').strip()
        if 'Lac' in p:
            return float(p.replace('Lacs', '').replace('Lac', '').strip()) * 100000
        return float(p)
    except:
        return None


In [3]:
def area_to_sqft(a):
    if not a:
        return None
    a = str(a).replace(',', '').strip()
    try:
        if 'Kanal' in a:
            return float(a.replace('Kanal', '').strip()) * 5400
        elif 'Marla' in a:
            return float(a.replace('Marla', '').strip()) * 272.25
        elif 'SQFT' in a or 'Sq. Ft.' in a:
            return float(re.sub(r'[^\d.]', '', a))
        elif 'SQYD' in a or 'Sq. Yd.' in a:
            return float(re.sub(r'[^\d.]', '', a)) * 9
        else:
            return float(a)
    except:
        return None


In [4]:
def parse_date(text):
    if pd.isna(text): # if no value in the tuple
        return None
    text = str(text).lower().strip() # convert text to string and lowercase
    now  = datetime.now() #gives current date and time so that we can use it in calcualting proper date for each listing
    try:
        if 'minute' in text: # like 13 minutes ago
            n = int(re.search(r'(\d+)', text).group(1)) #number extarcted using regex
            return (now - timedelta(minutes=n)).strftime('%Y-%m-%d')
        
        elif 'hour' in text: # like 2 hours ago
            n = int(re.search(r'(\d+)', text).group(1))
            return (now - timedelta(hours=n)).strftime('%Y-%m-%d')
        
        elif 'day' in text: # 3 days ago
            n = int(re.search(r'(\d+)', text).group(1))
            return (now - timedelta(days=n)).strftime('%Y-%m-%d')
        
        elif 'week' in text: # 1 week ago
            n = int(re.search(r'(\d+)', text).group(1))
            return (now - timedelta(weeks=n)).strftime('%Y-%m-%d')
        
        elif 'month' in text:
            n = int(re.search(r'(\d+)', text).group(1))
            return (now - timedelta(days=n * 30)).strftime('%Y-%m-%d')
        
        elif 'year' in text:
            n = int(re.search(r'(\d+)', text).group(1))
            return (now - timedelta(days=n * 365)).strftime('%Y-%m-%d')
        
        else:
            return None
    except:
        return None


In [5]:
print('Loading CSV: OLXdata_full1.csv')
df = pd.read_csv('OLXdata_full1.csv')  # raw data 

print(f'Rows loaded: {len(df)}')
print(f'Columns: {list(df.columns)}')
display(df.head())

Loading CSV: OLXdata_full1.csv
Rows loaded: 978
Columns: ['Title', 'Price', 'Location', 'Area', 'Bedrooms', 'Bathrooms', 'Date_Posted', 'Description', 'Seller_Name', 'Image_Count', 'URL', 'Label']


,Title,Price,Location,Area,Bedrooms,Bathrooms,Date_Posted,Description,Seller_Name,Image_Count,URL,Label
0,"Luxurious 2BHK In DHA Phase 8, Lahore","Rs 15,000","DHA Phase 8 - Ex Air Avenue, Lahore•",250 SQFT,2 Beds,2 Baths,1 day ago,NaN,NaN,0,https://www.olx.com.pk/item/luxurious-2bhk-in-...,NaN
1,Independent Banglow 120 Sq Yards Double Story ...,"Rs 68,000","Gulistan-e-Jauhar Block 7, Karachi•",120 SQYD,4 Beds,4 Baths,1 week ago,Description Independent House 120 Sq Yards Dou...,Ajwa Marketing,0,https://www.olx.com.pk/item/independent-banglo...,NaN
2,A Stunning Prime Location Flat Is Up For Grabs...,"Rs 8,000","Royal Palm City, Gujranwala•",3 Marla,2 Beds,2 Baths,2 days ago,Description Per Day Flat Available For Rent,Adeel Mehar,0,https://www.olx.com.pk/item/a-stunning-prime-l...,NaN
3,10 Marla House Fully Furnished For Rent Sector...,Rs 2.29 Lac,"Bahria Town - Jasmine Block, Lahore•",10 Marla,5 Beds,7 Baths,2 weeks ago,Description 10 Marla Luxurious Designer Full F...,Waqas,0,https://www.olx.com.pk/item/10-marla-house-ful...,NaN
4,1 Kanal Brand New Designer Double Storey House...,Rs 4.20 Lac,"DHA Defence Phase 2, Islamabad•",1 Kanal,6 Beds,6 Baths,15 hours ago,Description 1 kanal brand new double story hou...,Muhammad Sohail,0,https://www.olx.com.pk/item/1-kanal-brand-new-...,NaN


In [6]:
df_cleaned = df.copy()  # raw df as a copy and then clean it

# Price clean
print('Cleaning Price column...')
df_cleaned['Price'] = df_cleaned['Price'].apply(clean_price)

# Area to sqft
print('Converting Area to sqft...')
df_cleaned['Area_sqft'] = df_cleaned['Area'].apply(area_to_sqft)

# Date parse
print('Parsing Date_Posted...')
df_cleaned['Date_Posted'] = df_cleaned['Date_Posted'].apply(parse_date)

# Location clean
print('Cleaning Location...')
df_cleaned['Location'] = df_cleaned['Location'].str.replace('•', '').str.strip()

# Drop unnecessary
df_cleaned = df_cleaned.drop(columns=['Image_Count'], errors='ignore')

# Missing values fill
print('Filling missing values...')
df_cleaned['Description'] = df_cleaned['Description'].fillna('no description')
df_cleaned['Seller_Name'] = df_cleaned['Seller_Name'].fillna('Not specified')
df_cleaned['Area_sqft']   = df_cleaned['Area_sqft'].fillna(df_cleaned['Area_sqft'].median())

# Categorical encoding
df_cleaned['Bed_Count']  = df_cleaned['Bedrooms'].str.extract(r'(\d+)').astype(float).fillna(1)
df_cleaned['Bath_Count'] = df_cleaned['Bathrooms'].str.extract(r'(\d+)').astype(float).fillna(1)

df_cleaned = df_cleaned.reset_index(drop=True)

print('\nMissing values after cleaning:')
print(df_cleaned.isnull().sum())
display(df_cleaned.head(10))


Cleaning Price column...
Converting Area to sqft...
Parsing Date_Posted...
Cleaning Location...
Filling missing values...

Missing values after cleaning:
Title            0
Price            0
Location         0
Area             0
Bedrooms         0
Bathrooms        0
Date_Posted      0
Description      0
Seller_Name      0
URL              0
Label          978
Area_sqft        0
Bed_Count        0
Bath_Count       0
dtype: int64


,Title,Price,Location,Area,Bedrooms,Bathrooms,Date_Posted,Description,Seller_Name,URL,Label,Area_sqft,Bed_Count,Bath_Count
0,"Luxurious 2BHK In DHA Phase 8, Lahore",15000.0,"DHA Phase 8 - Ex Air Avenue, Lahore",250 SQFT,2 Beds,2 Baths,2026-04-18,no description,Not specified,https://www.olx.com.pk/item/luxurious-2bhk-in-...,NaN,250.00,2.0,2.0
1,Independent Banglow 120 Sq Yards Double Story ...,68000.0,"Gulistan-e-Jauhar Block 7, Karachi",120 SQYD,4 Beds,4 Baths,2026-04-12,Description Independent House 120 Sq Yards Dou...,Ajwa Marketing,https://www.olx.com.pk/item/independent-banglo...,NaN,1080.00,4.0,4.0
2,A Stunning Prime Location Flat Is Up For Grabs...,8000.0,"Royal Palm City, Gujranwala",3 Marla,2 Beds,2 Baths,2026-04-17,Description Per Day Flat Available For Rent,Adeel Mehar,https://www.olx.com.pk/item/a-stunning-prime-l...,NaN,816.75,2.0,2.0
3,10 Marla House Fully Furnished For Rent Sector...,229000.0,"Bahria Town - Jasmine Block, Lahore",10 Marla,5 Beds,7 Baths,2026-04-05,Description 10 Marla Luxurious Designer Full F...,Waqas,https://www.olx.com.pk/item/10-marla-house-ful...,NaN,2722.50,5.0,7.0
4,1 Kanal Brand New Designer Double Storey House...,420000.0,"DHA Defence Phase 2, Islamabad",1 Kanal,6 Beds,6 Baths,2026-04-18,Description 1 kanal brand new double story hou...,Muhammad Sohail,https://www.olx.com.pk/item/1-kanal-brand-new-...,NaN,5400.00,6.0,6.0
5,5 Marla Furnished House For Rent In Green City...,190000.0,"Green City, Lahore",5 Marla,3 Beds,4 Baths,2026-04-19,Description A Spacious House In Green City Is ...,AliZay Chaudhary,https://www.olx.com.pk/item/5-marla-furnished-...,NaN,1361.25,3.0,4.0
6,2 Bed Flat For Rent in Munawr chowrangi Block ...,30000.0,"Gulistan-e-Jauhar Block 12, Karachi",750 SQFT,2 Beds,2 Baths,2026-04-19,Description 2 Bed/Kitchen \nMan Munawr chowran...,Sardar Muhammad Asif,https://www.olx.com.pk/item/2-bed-flat-for-ren...,NaN,750.00,2.0,2.0
7,1 Kanal Ground Portion For Rent,160000.0,"DHA Defence Phase 2, Islamabad",1 Kanal,3 Beds,3 Baths,2026-04-19,Description 1 Kanal Corner House house Ground ...,Muhammad Qasim,https://www.olx.com.pk/item/1-kanal-ground-por...,NaN,5400.00,3.0,3.0
8,Beautiful First Floor House For Rent Near Expr...,32000.0,"Ghauri Town Phase 4B, Islamabad",5 Marla,2 Beds,2 Baths,2026-04-19,Description Beautiful First Floor House For Re...,Syed Muhammad Ibrahim,https://www.olx.com.pk/item/beautiful-first-fl...,NaN,1361.25,2.0,2.0
9,ONE KANAL PORTION FOR RENT,125000.0,"Sui Gas Society Phase 1, Lahore",1 Kanal,3 Beds,4 Baths,2026-04-19,Description Three bedrooms with attached bath ...,Bin Hatam Group,https://www.olx.com.pk/item/one-kanal-portion-...,NaN,5400.00,3.0,4.0


In [7]:
# now, Fraud Labeling Heuristic
# Convention: 1 = Fraud  0 = Legitimate

df_labelled = df_cleaned.copy() #copy of cleaned data

# RULE 1: Suspicious keywords in title or description 
FRAUD_KEYWORDS = [
    'urgent', 'immediately', 'hurry', 'limited time',
    'abroad', 'outside pakistan', 'currently not in pakistan',
    'owner abroad', 'not in country',
    'advance only', 'deposit only', 'send money',
    'western union', 'bank transfer only', 'keys on payment',
    'whatsapp only', 'no calls', 'sms only',
    'per day', 'daily basis', 'short time', 'per hour',
    'short stay', 'hourly',
    'god fearing', 'honest owner', 'serious buyer only',
    'no time wasters', 'no agents'
]

def keyword_fraud(row):
    title= str(row.get('Title', '')       or '').lower()
    desc = str(row.get('Description', '') or '').lower()
    combined = title + ' ' + desc
    return int(any(kw in combined for kw in FRAUD_KEYWORDS))
    # any() means agar koi bhi keyword mila toh True => int(True) = 1
    # koi nahi mila toh False => int(False) = 0


# RULE 2: Price anomaly
def price_anomaly(row):
    price= float(row.get('Price', 0) or 0)
    area = str(row.get('Area','') or '').lower()
    location = str(row.get('Location', '') or '').lower()

    if price < 5000:
        return 1 # 1. if price of any property < 5000 = fraud

    #2. let, dha, bahria, gulberg etc mentioned below are premimum areas
    # in areas mein if price less then fraud
    premium = any(x in location for x in ['dha', 'bahria', 'gulberg', 'f-7', 'f-6', 'f-8'])
    if premium and price < 20000:
        return 1

    if 'kanal' in area and price < 30000: #3. if ghar/property bhari but price less = fraud
        return 1

    if '10 marla' in area and price < 15000: #4.if 10 Marla price kam = fraud
        return 1

    return 0 # no rule applied then = not fraud/ legitimate


# RULE 3: Phone number hidden in description
def phone_in_desc(row):
    desc= str(row.get('Description', '') or '')

    #IN OLX there is often fraudster who wrte phone no. in description 
    # so removing the slashes and then identifying
    cleaned = desc.replace('/', '').replace('-', '').replace(' ', '')

    count= 0 #consecutive digit counter
    for ch in cleaned:
        if ch.isdigit():
            count += 1
            if count >= 10: #if 10 > digits then fraud
                return 1
        else:
            count = 0
    return 0


#RULE 4: Copy-paste / template / empty descriptions
# if we find below sentences in many lstings then we can say it by fraudsters
TEMPLATE_PHRASES = [
    'for any property sale and purchase contact me an other detail ha hn',
    'for grabs', 'hard to come by', 'make your decision quickly',
    'trust us', 'per day flat available for rent',
    'other option also available for rent short terms'
]

def template_desc(row):
    desc = str(row.get('Description', '') or '').lower()
    #if desription < 30 characters / or/ empty then 100% suspicious
    if len(desc.strip()) < 30:
        return 1
    if any(phrase in desc for phrase in TEMPLATE_PHRASES): #copy paste phrase
        return 1
    return 0


#RULE 5: Duplicate descriptions -> same text used 3+ times)
desc_counts = df_labelled['Description'].value_counts()
duplicate_descs = set(desc_counts[desc_counts > 2].index)

def duplicate_listing(row):
    return 1 if row.get('Description') in duplicate_descs else 0


#Now, all done -> Applying all above rules 

df_labelled['R1_Keyword']= df_labelled.apply(keyword_fraud,     axis=1)
df_labelled['R2_Price'] = df_labelled.apply(price_anomaly,     axis=1)
df_labelled['R3_PhoneDesc']= df_labelled.apply(phone_in_desc,     axis=1)
df_labelled['R4_Template']= df_labelled.apply(template_desc,     axis=1)
df_labelled['R5_Duplicate'] = df_labelled.apply(duplicate_listing, axis=1)

df_labelled['Fraud_Score'] = (
    df_labelled['R1_Keyword'] +
    df_labelled['R2_Price'] +
    df_labelled['R3_PhoneDesc'] +
    df_labelled['R4_Template'] +
    df_labelled['R5_Duplicate']
)

# Fraud if 2 or more rules are correct/triggered
df_labelled['Label'] = (df_labelled['Fraud_Score'] >= 2).astype(int)

# for anaylsis displaying
total = len(df_labelled)
fraud = (df_labelled['Label'] == 1).sum()
legit = (df_labelled['Label'] == 0).sum()

print('\nLABELLING SUMMARY')
print('_______________________________________________________')
print(f'Total listings : {total}')
print(f'Fraud  (1): {fraud}  ({fraud/total*100:.1f}%)')
print(f'Legit  (0): {legit}  ({legit/total*100:.1f}%)')
print()
print('Rules triggered:')
print(f'R1 Keyword flags : {df_labelled["R1_Keyword"].sum()}')
print(f'R2 Price anomaly : {df_labelled["R2_Price"].sum()}')
print(f'R3 Phone in desc : {df_labelled["R3_PhoneDesc"].sum()}')
print(f'R4 Template desc : {df_labelled["R4_Template"].sum()}')
print(f'R5 Duplicate : {df_labelled["R5_Duplicate"].sum()}')


LABELLING SUMMARY
_______________________________________________________
Total listings : 978
Fraud  (1): 327  (33.4%)
Legit  (0): 651  (66.6%)

Rules triggered:
R1 Keyword flags : 245
R2 Price anomaly : 295
R3 Phone in desc : 7
R4 Template desc : 75
R5 Duplicate : 341


In [8]:
print('Listings flagged as Fraud (Label=1):')
flagged = df_labelled[df_labelled['Label'] == 1][
    ['Title', 'Price', 'Location', 'Seller_Name',
     'Fraud_Score', 'R1_Keyword', 'R2_Price',
     'R3_PhoneDesc', 'R4_Template', 'R5_Duplicate']
]
display(flagged)
print(f'Total fraud candidates: {len(flagged)}')

Listings flagged as Fraud (Label=1):


,Title,Price,Location,Seller_Name,Fraud_Score,R1_Keyword,R2_Price,R3_PhoneDesc,R4_Template,R5_Duplicate
0,"Luxurious 2BHK In DHA Phase 8, Lahore",15000.0,"DHA Phase 8 - Ex Air Avenue, Lahore",Not specified,3,0,1,0,1,1
2,A Stunning Prime Location Flat Is Up For Grabs...,8000.0,"Royal Palm City, Gujranwala",Adeel Mehar,2,1,0,0,1,0
10,1 kanal house luxury for rent,350000.0,"Garden Town, Lahore",Mirza Rizwan brand,2,0,0,0,1,1
15,10 marla house for rent,200000.0,"Upper Mall, Lahore",Mirza Rizwan brand,2,0,0,0,1,1
21,Apartment On Daily Basis furnished - Islamabad...,6000.0,"Capital Heights, Islamabad",Afraz Shaheen,2,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...
953,Bahria twon empair hait phase 6,3000.0,"Bahria Town, Islamabad",Asad khan,3,1,1,0,0,1
960,appartment for rent on daily and weekly baises,8000.0,"Bahria Heights 1, Rawalpindi",Not specified,3,0,1,0,1,1
966,Double Bed Furnished Flat Available for Rent (...,4980.0,"Citi Housing Society, Gujranwala",Ilyas Idrees,2,1,1,0,0,0
972,Double Bed Furnished Flat Available for Rent (...,4890.0,"Citi Housing Society, Gujranwala",Ilyas Idrees,2,1,1,0,0,0


Total fraud candidates: 327


In [64]:
# complete csv with rules versions also
df_labelled.to_csv('OLXdata_labelled.csv', index=False, encoding='utf-8-sig')
print('OLXdata_labelled.csv saved!')

"""
# Clean version ->  original columns + Label only (no rules)
clean_cols = ['Title', 'Price', 'Location', 'Area', 'Bedrooms',
              'Bathrooms', 'Date_Posted', 'Description',
              'Seller_Name', 'Area_sqft', 'URL', 'Label']

clean_cols = [c for c in clean_cols if c in df_labelled.columns]

df_labelled[clean_cols].to_csv('OLXdata_clean.csv', index=False, encoding='utf-8-sig')
print('OLXdata_clean.csv saved (original columns + Label)!')
print()
"""


OLXdata_labelled.csv saved!


"\n# Clean version ->  original columns + Label only (no rules)\nclean_cols = ['Title', 'Price', 'Location', 'Area', 'Bedrooms',\n              'Bathrooms', 'Date_Posted', 'Description',\n              'Seller_Name', 'Area_sqft', 'URL', 'Label']\n\nclean_cols = [c for c in clean_cols if c in df_labelled.columns]\n\ndf_labelled[clean_cols].to_csv('OLXdata_clean.csv', index=False, encoding='utf-8-sig')\nprint('OLXdata_clean.csv saved (original columns + Label)!')\nprint()\n"

In [65]:
df_labelled.head(10)

,Title,Price,Location,Area,Bedrooms,Bathrooms,Date_Posted,Description,Seller_Name,URL,Label,Area_sqft,Bed_Count,Bath_Count,R1_Keyword,R2_Price,R3_PhoneDesc,R4_Template,R5_Duplicate,Fraud_Score
0,"Luxurious 2BHK In DHA Phase 8, Lahore",15000.0,"DHA Phase 8 - Ex Air Avenue, Lahore",250 SQFT,2 Beds,2 Baths,2026-03-14,no description,Not specified,https://www.olx.com.pk/item/luxurious-2bhk-in-...,1,250.00,2.0,2.0,0,1,0,1,1,3
1,Independent Banglow 120 Sq Yards Double Story ...,68000.0,"Gulistan-e-Jauhar Block 7, Karachi",120 SQYD,4 Beds,4 Baths,2026-03-08,Description Independent House 120 Sq Yards Dou...,Ajwa Marketing,https://www.olx.com.pk/item/independent-banglo...,0,1080.00,4.0,4.0,0,0,1,0,0,1
2,A Stunning Prime Location Flat Is Up For Grabs...,8000.0,"Royal Palm City, Gujranwala",3 Marla,2 Beds,2 Baths,2026-03-13,Description Per Day Flat Available For Rent,Adeel Mehar,https://www.olx.com.pk/item/a-stunning-prime-l...,1,816.75,2.0,2.0,1,0,0,1,0,2
3,10 Marla House Fully Furnished For Rent Sector...,229000.0,"Bahria Town - Jasmine Block, Lahore",10 Marla,5 Beds,7 Baths,2026-03-01,Description 10 Marla Luxurious Designer Full F...,Waqas,https://www.olx.com.pk/item/10-marla-house-ful...,0,2722.50,5.0,7.0,0,0,0,0,0,0
4,1 Kanal Brand New Designer Double Storey House...,420000.0,"DHA Defence Phase 2, Islamabad",1 Kanal,6 Beds,6 Baths,2026-03-14,Description 1 kanal brand new double story hou...,Muhammad Sohail,https://www.olx.com.pk/item/1-kanal-brand-new-...,0,5400.00,6.0,6.0,0,0,0,0,0,0
5,5 Marla Furnished House For Rent In Green City...,190000.0,"Green City, Lahore",5 Marla,3 Beds,4 Baths,2026-03-15,Description A Spacious House In Green City Is ...,AliZay Chaudhary,https://www.olx.com.pk/item/5-marla-furnished-...,0,1361.25,3.0,4.0,0,0,0,1,0,1
6,2 Bed Flat For Rent in Munawr chowrangi Block ...,30000.0,"Gulistan-e-Jauhar Block 12, Karachi",750 SQFT,2 Beds,2 Baths,2026-03-15,Description 2 Bed/Kitchen \nMan Munawr chowran...,Sardar Muhammad Asif,https://www.olx.com.pk/item/2-bed-flat-for-ren...,0,750.00,2.0,2.0,0,0,0,0,0,0
7,1 Kanal Ground Portion For Rent,160000.0,"DHA Defence Phase 2, Islamabad",1 Kanal,3 Beds,3 Baths,2026-03-15,Description 1 Kanal Corner House house Ground ...,Muhammad Qasim,https://www.olx.com.pk/item/1-kanal-ground-por...,0,5400.00,3.0,3.0,0,0,0,0,0,0
8,Beautiful First Floor House For Rent Near Expr...,32000.0,"Ghauri Town Phase 4B, Islamabad",5 Marla,2 Beds,2 Baths,2026-03-15,Description Beautiful First Floor House For Re...,Syed Muhammad Ibrahim,https://www.olx.com.pk/item/beautiful-first-fl...,0,1361.25,2.0,2.0,0,0,0,0,0,0
9,ONE KANAL PORTION FOR RENT,125000.0,"Sui Gas Society Phase 1, Lahore",1 Kanal,3 Beds,4 Baths,2026-03-15,Description Three bedrooms with attached bath ...,Bin Hatam Group,https://www.olx.com.pk/item/one-kanal-portion-...,0,5400.00,3.0,4.0,0,0,0,0,0,0


In [66]:
print(df_labelled.isnull().sum())


Title           0
Price           0
Location        0
Area            0
Bedrooms        0
Bathrooms       0
Date_Posted     0
Description     0
Seller_Name     0
URL             0
Label           0
Area_sqft       0
Bed_Count       0
Bath_Count      0
R1_Keyword      0
R2_Price        0
R3_PhoneDesc    0
R4_Template     0
R5_Duplicate    0
Fraud_Score     0
dtype: int64


In [68]:
df_labelled.head(8)

,Title,Price,Location,Area,Bedrooms,Bathrooms,Date_Posted,Description,Seller_Name,URL,Label,Area_sqft,Bed_Count,Bath_Count,R1_Keyword,R2_Price,R3_PhoneDesc,R4_Template,R5_Duplicate,Fraud_Score
0,"Luxurious 2BHK In DHA Phase 8, Lahore",15000.0,"DHA Phase 8 - Ex Air Avenue, Lahore",250 SQFT,2 Beds,2 Baths,2026-03-14,no description,Not specified,https://www.olx.com.pk/item/luxurious-2bhk-in-...,1,250.00,2.0,2.0,0,1,0,1,1,3
1,Independent Banglow 120 Sq Yards Double Story ...,68000.0,"Gulistan-e-Jauhar Block 7, Karachi",120 SQYD,4 Beds,4 Baths,2026-03-08,Description Independent House 120 Sq Yards Dou...,Ajwa Marketing,https://www.olx.com.pk/item/independent-banglo...,0,1080.00,4.0,4.0,0,0,1,0,0,1
2,A Stunning Prime Location Flat Is Up For Grabs...,8000.0,"Royal Palm City, Gujranwala",3 Marla,2 Beds,2 Baths,2026-03-13,Description Per Day Flat Available For Rent,Adeel Mehar,https://www.olx.com.pk/item/a-stunning-prime-l...,1,816.75,2.0,2.0,1,0,0,1,0,2
3,10 Marla House Fully Furnished For Rent Sector...,229000.0,"Bahria Town - Jasmine Block, Lahore",10 Marla,5 Beds,7 Baths,2026-03-01,Description 10 Marla Luxurious Designer Full F...,Waqas,https://www.olx.com.pk/item/10-marla-house-ful...,0,2722.50,5.0,7.0,0,0,0,0,0,0
4,1 Kanal Brand New Designer Double Storey House...,420000.0,"DHA Defence Phase 2, Islamabad",1 Kanal,6 Beds,6 Baths,2026-03-14,Description 1 kanal brand new double story hou...,Muhammad Sohail,https://www.olx.com.pk/item/1-kanal-brand-new-...,0,5400.00,6.0,6.0,0,0,0,0,0,0
5,5 Marla Furnished House For Rent In Green City...,190000.0,"Green City, Lahore",5 Marla,3 Beds,4 Baths,2026-03-15,Description A Spacious House In Green City Is ...,AliZay Chaudhary,https://www.olx.com.pk/item/5-marla-furnished-...,0,1361.25,3.0,4.0,0,0,0,1,0,1
6,2 Bed Flat For Rent in Munawr chowrangi Block ...,30000.0,"Gulistan-e-Jauhar Block 12, Karachi",750 SQFT,2 Beds,2 Baths,2026-03-15,Description 2 Bed/Kitchen \nMan Munawr chowran...,Sardar Muhammad Asif,https://www.olx.com.pk/item/2-bed-flat-for-ren...,0,750.00,2.0,2.0,0,0,0,0,0,0
7,1 Kanal Ground Portion For Rent,160000.0,"DHA Defence Phase 2, Islamabad",1 Kanal,3 Beds,3 Baths,2026-03-15,Description 1 Kanal Corner House house Ground ...,Muhammad Qasim,https://www.olx.com.pk/item/1-kanal-ground-por...,0,5400.00,3.0,3.0,0,0,0,0,0,0
